In [0]:
# Imports

from pyspark.sql.functions import (
    col,
    count,
    when,
    upper,
    trim,
    regexp_replace,
    round as spark_round
)

In [0]:
# table configuration

bronze_table = "online_retail.bronze.transactions_raw"
silver_table = "online_retail.silver.transactions_clean"

bronze_df = spark.table(bronze_table)

bronze_row_count = bronze_df.count()

print(f"Bronze rows: {bronze_row_count:,}")

Bronze rows: 45,228


In [0]:
# Remove exact business duplicates

business_columns = [
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "price",
    "customer_id",
    "country"
]

deduplicated_df = bronze_df.dropDuplicates(
    business_columns
)

deduplicated_row_count = deduplicated_df.count()

duplicate_count = (
    bronze_row_count - deduplicated_row_count
)

print(f"Duplicate rows removed: {duplicate_count:,}")

print(
    f"Rows after deduplication: "
    f"{deduplicated_row_count:,}"
)

Duplicate rows removed: 506
Rows after deduplication: 44,722


In [0]:
# Profile source quality after deduplication

quality_profile_df = deduplicated_df.agg(
    count(
        when(col("invoice").isNull(), 1)
    ).alias("missing_invoice"),

    count(
        when(col("stock_code").isNull(), 1)
    ).alias("missing_stock_code"),

    count(
        when(col("description").isNull(), 1)
    ).alias("missing_description"),

    count(
        when(col("quantity").isNull(), 1)
    ).alias("missing_quantity"),

    count(
        when(col("invoice_date").isNull(), 1)
    ).alias("missing_invoice_date"),

    count(
        when(col("price").isNull(), 1)
    ).alias("missing_price"),

    count(
        when(col("customer_id").isNull(), 1)
    ).alias("missing_customer_id"),

    count(
        when(col("country").isNull(), 1)
    ).alias("missing_country"),

    count(
        when(col("quantity") < 0, 1)
    ).alias("negative_quantity"),

    count(
        when(col("quantity") == 0, 1)
    ).alias("zero_quantity"),

    count(
        when(col("price") < 0, 1)
    ).alias("negative_price"),

    count(
        when(col("price") == 0, 1)
    ).alias("zero_price"),

    count(
        when(
            upper(col("invoice")).startswith("C"),
            1
        )
    ).alias("cancelled_invoices")
)

quality_profile_df.show(truncate=False)

+---------------+------------------+-------------------+----------------+--------------------+-------------+-------------------+---------------+-----------------+-------------+--------------+----------+------------------+
|missing_invoice|missing_stock_code|missing_description|missing_quantity|missing_invoice_date|missing_price|missing_customer_id|missing_country|negative_quantity|zero_quantity|negative_price|zero_price|cancelled_invoices|
+---------------+------------------+-------------------+----------------+--------------------+-------------+-------------------+---------------+-----------------+-------------+--------------+----------+------------------+
|0              |0                 |228                |0               |0                   |0            |13446              |0              |1103             |0            |0             |256       |1013              |
+---------------+------------------+-------------------+----------------+--------------------+-------------+----

In [0]:
# Clean text fields and customer IDs

cleaned_df = (
    deduplicated_df
    .withColumn(
        "invoice",
        trim(col("invoice"))
    )
    .withColumn(
        "stock_code",
        trim(col("stock_code"))
    )
    .withColumn(
        "description",
        trim(col("description"))
    )
    .withColumn(
        "customer_id",
        regexp_replace(
            trim(col("customer_id")),
            r"\.0$",
            ""
        )
    )
    .withColumn(
        "country",
        trim(col("country"))
    )
)

cleaned_df_row_count = cleaned_df.count()

print(f"Cleaned rows: {cleaned_df_row_count:,}")

print(
    "Cleaning preserved all deduplicated rows:",
    cleaned_df_row_count == deduplicated_row_count
)

(
    cleaned_df
    .filter(col("customer_id").isNotNull())
    .select("customer_id")
    .show(5, truncate=False)
)

Cleaned rows: 44,722
Cleaning preserved all deduplicated rows: True
+-----------+
|customer_id|
+-----------+
|13085      |
|13085      |
|13085      |
|13085      |
|13085      |
+-----------+
only showing top 5 rows


In [0]:
# Add transaction and quality flags

silver_df = (
    cleaned_df
    .withColumn(
        "is_cancelled",
        when(
            upper(col("invoice")).startswith("C"),
            True
        ).otherwise(False)
    )
    .withColumn(
        "has_customer_id",
        when(
            col("customer_id").isNotNull()
            & (col("customer_id") != ""),
            True
        ).otherwise(False)
    )
    .withColumn(
        "has_description",
        when(
            col("description").isNotNull()
            & (col("description") != ""),
            True
        ).otherwise(False)
    )
    .withColumn(
        "is_positive_sale",
        when(
            (~col("is_cancelled"))
            & (col("quantity") > 0)
            & (col("price") > 0),
            True
        ).otherwise(False)
    )
    .withColumn(
        "line_total",
        spark_round(
            col("quantity") * col("price"),
            2
        )
    )
)

In [0]:
# Validate Silver transformations

silver_row_count = silver_df.count()

print(f"Silver rows before write: {silver_row_count:,}")

print(
    "Silver count matches deduplicated count:",
    silver_row_count == deduplicated_row_count
)

silver_profile_df = silver_df.agg(
    count(
        when(col("is_cancelled"), 1)
    ).alias("cancelled_records"),

    count(
        when(col("is_positive_sale"), 1)
    ).alias("positive_sale_records"),

    count(
        when(~col("has_customer_id"), 1)
    ).alias("missing_customer_id_records"),

    count(
        when(~col("has_description"), 1)
    ).alias("missing_description_records")
)

silver_profile_df.show(truncate=False)

Silver rows before write: 44,722
Silver count matches deduplicated count: True
+-----------------+---------------------+---------------------------+---------------------------+
|cancelled_records|positive_sale_records|missing_customer_id_records|missing_description_records|
+-----------------+---------------------+---------------------------+---------------------------+
|1013             |43453                |13446                      |228                        |
+-----------------+---------------------+---------------------------+---------------------------+



In [0]:
# Write the managed Silver Delta table

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
# Validate the stored Silver table

stored_silver_df = spark.table(silver_table)

stored_silver_row_count = stored_silver_df.count()

print(f"Stored Silver rows: {stored_silver_row_count:,}")

print(
    "Stored count matches Silver DataFrame:",
    stored_silver_row_count == silver_row_count
)

(
    stored_silver_df
    .select(
        "invoice",
        "customer_id",
        "quantity",
        "price",
        "is_cancelled",
        "is_positive_sale",
        "line_total"
    )
    .show(5, truncate=False)
)

Stored Silver rows: 44,722
Stored count matches Silver DataFrame: True
+-------+-----------+--------+-----+------------+----------------+----------+
|invoice|customer_id|quantity|price|is_cancelled|is_positive_sale|line_total|
+-------+-----------+--------+-----+------------+----------------+----------+
|489435 |13085      |12      |3.75 |false       |true            |45.0      |
|489439 |12682      |25      |0.42 |false       |true            |10.5      |
|489439 |12682      |12      |1.65 |false       |true            |19.8      |
|489440 |18087      |8       |2.55 |false       |true            |20.4      |
|489445 |17519      |12      |0.85 |false       |true            |10.2      |
+-------+-----------+--------+-----+------------+----------------+----------+
only showing top 5 rows
